# 04 – Metadata Agent (Collibra DGC)

The **MetadataAgent** retrieves data asset metadata and DQ scores from Collibra.  
Mock mode returns canned assets — no Collibra credentials needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.metadata_agent import MetadataAgent
from core.base_agent import AgentRequest

agent = MetadataAgent()

## 1. Fetch metadata for a single product

In [ ]:
req = AgentRequest(query='What is the data quality score for retention?', data_products=['retention'])
result = agent.execute(req)

print('Success   :', result.success)
print('Message   :', result.message)
print('Confidence:', result.confidence)

meta = result.data.get('retention', {})
print('\nRetention asset metadata:')
for k, v in meta.items():
    print(f'  {k}: {v}')

## 2. All four products

In [ ]:
req_all = AgentRequest(
    query='Show metadata for all data products',
    data_products=['retention', 'bookings', 'cac', 'ltv'],
)
result = agent.execute(req_all)

print(f"Assets found: {result.metadata['assets_found']}\n")

for product, meta in result.data.items():
    dq = meta.get('data_quality', {})
    print(f"  {product.upper()}")
    print(f"    owner  : {meta.get('owner')}")
    print(f"    domain : {meta.get('domain')}")
    print(f"    status : {meta.get('status')}")
    print(f"    DQ score: {dq.get('score')}%  ({dq.get('passed')}/{dq.get('total_rules')} rules passed)")
    print()

## 3. Keyword alias resolution

In [ ]:
# The agent maps synonyms to canonical product names
alias_queries = [
    'Who owns the churn dataset?',      # churn -> retention
    'What is the ARR data quality?',    # arr -> bookings
    'Show me acquisition cost metadata',# acquisition -> cac
    'Lifetime value asset info',        # lifetime -> ltv
]

for q in alias_queries:
    r = agent.execute(AgentRequest(query=q))
    products_found = list(r.data.keys())
    print(f"'{q}'")
    print(f"  => resolved products: {products_found}")
    print()

## 4. Direct mock service inspection

In [ ]:
from services.collibra.mock import MockCollibraService

svc = MockCollibraService()

# search_assets
assets = svc.search_assets('bookings')
print('search_assets(bookings):', assets)

# get_data_quality
dq = svc.get_data_quality('asset-002')
print('\nget_data_quality(asset-002):', dq)

## 5. DQ score summary across all assets

In [ ]:
from services.collibra.mock import _ASSETS, _DQ

print(f"{'Product':<20} {'DQ Score':>10} {'Passed':>8} {'Failed':>8} {'Total':>8}")
print('-' * 56)
for asset_id, asset in _ASSETS.items():
    dq = _DQ.get(asset_id, {})
    score = dq.get('score', 0)
    flag = '  <<< BELOW 90%' if score < 90 else ''
    print(f"{asset['name']:<20} {score:>9}% {dq.get('passed',0):>8} {dq.get('failed',0):>8} {dq.get('total_rules',0):>8}{flag}")